# Rescoring the KALE results (E4, E5, replication) with a fixed scorer

All KALE scores computed in Colab so far (the replication table, E4, E5 and every validation table)
compared each caption with **the next painting's reference captions**.

**Why.** pycocoevalcap's tokenizer runs a Java program and matches its output lines to captions *by
position*. On Colab, Java printed **two extra warning lines** into that output, so every caption was
scored against the references of a painting two places away (section 2 reproduces the reported numbers).
The same extra lines made METEOR crash ("could not convert string to float"). The Qwen notebooks were not affected
(rescoring `data/baseline` reproduces the team's numbers exactly).

**Fix.** Start Java with logging switched off (the same flags `genai_vlm_E3_clean.ipynb` uses for
METEOR), check the tokenizer on three known captions, and check every BLEU-1 against a plain-Python
BLEU-1 that does not use Java at all.

This notebook only reads caption files that the training notebooks already saved to Drive; nothing is
retrained. A T4 is enough (BERTScore is the only part that uses the GPU).

1. Show the extra Java line, and the fix
2. Proof: reproducing the shift gives exactly the numbers Colab reported
3. E4: rescore the 8 validation epochs and re-pick the epoch
4. E5: rescore every DPO run and epoch on validation and re-pick
5. Test: E4 vs E5 with all metrics (BLEU, METEOR, ROUGE-L, CIDEr, BERTScore)
6. The replication run, rescored, vs the KALE paper

## Settings

In [ ]:
DRIVE_DIR = "/content/drive/MyDrive/KALE_files"
SFT_DIR = f'{DRIVE_DIR}/results_sft_full'      # saved by kale_retrain_colab.ipynb (E4)
DPO_DIR = f'{DRIVE_DIR}/results_dpo'           # saved by kale_dpo_colab.ipynb (E5)
OUT_DIR = f'{DRIVE_DIR}/results_rescored'      # corrected tables are written here
MIN_DISTINCT = 50                              # same diversity rule as the training notebooks
E4_PICKED_EPOCH = 5                            # what kale_retrain_colab.ipynb picked with the broken scores
E5_PICKED = ('b0.1_lr1e-6', 1)                 # what kale_dpo_colab.ipynb picked (run, epoch)
PAPER = {'Bleu_1': 59.78, 'Bleu_4': 20.14, 'METEOR': 19.39, 'ROUGE_L': 42.22, 'CIDEr': 36.51}   # KALE paper, ArtCap test

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, json, glob, re, subprocess, tempfile, urllib.request
if not os.path.exists('/content/emm'):
    !git clone -q https://github.com/emmadionne1/10623_Project_EMMbeddings.git /content/emm
else:
    !git -C /content/emm pull -q
%cd /content/emm
!pip install -q pycocoevalcap==1.2 bert-score
!which java > /dev/null || apt-get install -y -qq default-jre-headless > /dev/null
!java -version 2>&1 | head -1
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (BERTScore will be slow)')
os.makedirs(OUT_DIR, exist_ok=True)
for d in (SFT_DIR, DPO_DIR):
    assert os.path.isdir(d), f'{d} not found: run the training notebooks first'

## 1. The extra Java line, and the fix

This runs the exact Java command pycocoevalcap uses, on 3 captions. Correct output is 3 lines, in
order. Any line *before* `a cat , sitting .` shifts every caption after it.

Whether the warnings appear depends on the Colab machine and Java version, so this cell may show 3
lines both times. Section 2 shows the effect on the scores that were actually reported, and every
score below is checked against a plain-Python BLEU-1 either way.

In [ ]:
import pycocoevalcap.tokenizer.ptbtokenizer as ptb
JAR_DIR = os.path.dirname(ptb.__file__)
JAVA_FIX = '-XX:+IgnoreUnrecognizedVMOptions -XX:-UseContainerSupport -Xlog:all=off'

def raw_tokenizer_output(captions, java_options):
    """Run the Java command pycocoevalcap runs and return its output lines."""
    env = {k: v for k, v in os.environ.items() if k != 'JAVA_TOOL_OPTIONS'}
    if java_options:
        env['JAVA_TOOL_OPTIONS'] = java_options
    with tempfile.NamedTemporaryFile('w', delete=False, dir=JAR_DIR) as f:
        f.write('\n'.join(captions))
    cmd = ['java', '-cp', 'stanford-corenlp-3.4.1.jar', 'edu.stanford.nlp.process.PTBTokenizer',
           '-preserveLines', '-lowerCase', os.path.basename(f.name)]
    out = subprocess.run(cmd, cwd=JAR_DIR, env=env, capture_output=True, text=True).stdout
    os.remove(f.name)
    return out.rstrip('\n').split('\n')

caps = ['A cat, sitting.', 'two dogs', 'a red boat']
for label, opts in [('without the fix', None), ('with the fix   ', JAVA_FIX)]:
    lines = raw_tokenizer_output(caps, opts)
    print(f'{label}: {len(lines)} lines for {len(caps)} captions -> {lines}')

Apply the fix for everything below: `JAVA_TOOL_OPTIONS` is read by every Java program this notebook
starts (the tokenizer and METEOR). Then check the tokenizer on known captions. If Java does not
start, pycocoevalcap silently returns empty captions, so the check also catches that.

In [ ]:
os.environ['JAVA_TOOL_OPTIONS'] = JAVA_FIX
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
probe = PTBTokenizer().tokenize({0: [{'caption': 'A cat, sitting.'}], 1: [{'caption': 'two dogs'}],
                                 2: [{'caption': 'x'}, {'caption': 'y z'}]})
assert probe == {0: ['a cat sitting'], 1: ['two dogs'], 2: ['x', 'y z']}, f'tokenizer still misaligned: {probe}'
print('tokenizer alignment check passed')

# METEOR also needs its paraphrase table, which the pip package leaves out (same as genai_vlm_E3_clean.ipynb)
import pycocoevalcap.meteor.meteor as meteor_mod
para = os.path.join(os.path.dirname(meteor_mod.__file__), 'data', 'paraphrase-en.gz')
os.makedirs(os.path.dirname(para), exist_ok=True)
if not os.path.exists(para) or os.path.getsize(para) == 0:
    urllib.request.urlretrieve('https://raw.githubusercontent.com/tylin/coco-caption/master/pycocoevalcap/meteor/data/paraphrase-en.gz', para)
print('METEOR paraphrase table:', round(os.path.getsize(para) / 1e6, 1), 'MB')

Scoring uses the team's `helpers/metrics.py`, so KALE and Qwen numbers come from the same code.
Everything is shown x100. `distinct` is the % of captions that differ from every other caption.

In [ ]:
import pandas as pd
import helpers.metrics as hm
from helpers.metrics import tokenize_captions, compute_bleu, compute_rouge_l, compute_cider, compute_metrics

def load(path):
    res = json.load(open(path))
    preds = [' '.join(str(r['pred_caption']).split()) or 'none' for r in res]
    refs = [list(r['gold_caption']) for r in res]
    return preds, refs

def extras(preds):
    return {'mean_len': sum(len(p.split()) for p in preds) / len(preds),
            'distinct': 100 * len({p.lower() for p in preds}) / len(preds), 'n': len(preds)}

import collections, math

def plain_bleu1(preds, refs):
    """BLEU-1 in plain Python (no Java, no pycocoevalcap). If it disagrees with pycocoevalcap's Bleu_1,
    the Java tokenizer has paired captions with the wrong references."""
    words = lambda t: re.findall(r"[a-z0-9]+", t.lower())
    match = total = reflen = 0
    for p, rs in zip(preds, refs):
        pw, rws = words(p), [words(r) for r in rs]
        best = collections.Counter()
        for rw in rws:
            for w, n in collections.Counter(rw).items():
                best[w] = max(best[w], n)
        match += sum(min(n, best[w]) for w, n in collections.Counter(pw).items())
        total += len(pw)
        reflen += min((abs(len(rw) - len(pw)), len(rw)) for rw in rws)[1]
    return 100 * (1 if total > reflen else math.exp(1 - reflen / total)) * match / total

def check(s, preds, refs):
    b1 = plain_bleu1(preds, refs)
    assert abs(b1 - s['Bleu_1']) < 1.5, (f"Bleu_1 {s['Bleu_1']:.2f} but plain-Python BLEU-1 {b1:.2f}: "
                                         "captions are being scored against the wrong references")
    return s

def quick_scores(preds, refs, verify=True):
    """BLEU, ROUGE-L, CIDEr: enough to compare epochs on validation."""
    gen, exp = tokenize_captions(preds, refs)
    s = {k: 100 * v for k, v in {**compute_bleu(gen, exp), **compute_rouge_l(gen, exp), **compute_cider(gen, exp)}.items()}
    return check(s, preds, refs) if verify else s

def full_scores(preds, refs):
    """All metrics, including METEOR and BERTScore (helpers.metrics.compute_metrics)."""
    s = compute_metrics(preds, refs, device='cuda' if torch.cuda.is_available() else 'cpu')
    return check({k: 100 * v for k, v in s.items()}, preds, refs)

COLS = ['Bleu_1', 'Bleu_4', 'ROUGE_L', 'CIDEr', 'mean_len', 'distinct']

## 2. Proof: two extra lines reproduce what Colab reported

`ShiftedPTB` simulates the bug: `EXTRA` lines arrive before the first caption, so caption *k* receives
the tokens of caption *k - EXTRA*. With 2 extra lines, the replication run's epoch-0 test captions get
the numbers reported in Colab (`data/kale/kale_scores_by_epoch.csv`): ROUGE-L exactly, the rest within a
few tenths (we don't know the warning lines' exact words, which are scored as captions too).

In [ ]:
EXTRA = 2   # extra lines Java printed on Colab

class ShiftedPTB(PTBTokenizer):
    """The Colab bug: EXTRA lines before the first caption, so every caption gets the tokens EXTRA lines earlier."""
    def tokenize(self, captions_for_image):
        ok = super().tokenize(captions_for_image)
        keys = [k for k, v in captions_for_image.items() for _ in v]
        flat = (['java warning line'] * EXTRA + [t for k in captions_for_image for t in ok[k]])[:len(keys)]
        out = {}
        for k, t in zip(keys, flat):
            out.setdefault(k, []).append(t)
        return out

def scores_with_bug(preds, refs):
    hm.PTBTokenizer = ShiftedPTB      # tokenize_captions looks PTBTokenizer up in helpers.metrics
    try:
        return quick_scores(preds, refs, verify=False)
    finally:
        hm.PTBTokenizer = PTBTokenizer

preds, refs = load('data/kale/result_epoch0.json')
colab = pd.read_csv('data/kale/kale_scores_by_epoch.csv').drop_duplicates('epoch').set_index('epoch').loc[0]
proof = pd.DataFrame({
    'fixed scorer': quick_scores(preds, refs),
    f'bug reproduced ({EXTRA} extra lines)': scores_with_bug(preds, refs),
    'reported in Colab': {'Bleu_1': colab['B1'], 'Bleu_2': colab['B2'], 'Bleu_3': colab['B3'],
                          'Bleu_4': colab['B4'], 'ROUGE_L': colab['ROUGE_L'], 'CIDEr': colab['CIDEr']},
}).loc[['Bleu_1', 'Bleu_4', 'ROUGE_L', 'CIDEr']]
print('Replication run, epoch 0, 1082 test paintings')
print(proof.round(2).to_string())

# The bug also made a model that writes one caption for everything look good
one = ['a man and a woman are sitting at a table'] * len(refs)
print('\nCIDEr                       fixed   with bug')
print(f'replication epoch 0       {quick_scores(preds, refs)["CIDEr"]:7.1f}  {scores_with_bug(preds, refs)["CIDEr"]:8.1f}')
print(f'one caption for all       {quick_scores(one, refs)["CIDEr"]:7.1f}  {scores_with_bug(one, refs)["CIDEr"]:8.1f}')

## 3. E4: rescore the validation epochs and re-pick

Same rule as before: among epochs with at least `MIN_DISTINCT`% distinct captions, pick the highest
val CIDEr. The training notebook picked epoch 5 using the broken scores.

In [ ]:
files = {int(m.group(1)): f for f in glob.glob(f'{SFT_DIR}/result_epoch*.json')
         if (m := re.fullmatch(r'result_epoch(\d+)\.json', os.path.basename(f)))}
rows = {}
for e in sorted(files):
    p, r = load(files[e])
    rows[e] = {**quick_scores(p, r), **extras(p)}
e4_val = pd.DataFrame(rows).T[COLS]
e4_val.index.name = 'epoch'
e4_val['eligible'] = e4_val['distinct'] >= MIN_DISTINCT
e4_val.to_csv(f'{OUT_DIR}/e4_val_by_epoch.csv')
print(e4_val.round(2).to_string())

E4_BEST = int(e4_val[e4_val['eligible']]['CIDEr'].idxmax())
print(f'\nbest eligible epoch (fixed scores): {E4_BEST}   | picked earlier with broken scores: {E4_PICKED_EPOCH}')
print(f'best by val CIDEr alone, ignoring the diversity rule: {int(e4_val["CIDEr"].idxmax())}')
if E4_BEST != E4_PICKED_EPOCH:
    print(f'-> A different epoch wins. Only epoch {E4_PICKED_EPOCH} was saved, so using epoch {E4_BEST} '
          'needs the retrain notebook again (KALE uses a fixed seed, so its epochs repeat).')

## 4. E5: rescore every DPO run and epoch on validation

The DPO pair metrics (`val_loss`, `val_acc`) come from model probabilities, not from the tokenizer,
so they were never affected; only the caption scores change. Epoch -1 is the starting E4 checkpoint.

In [ ]:
runs = sorted(d for d in os.listdir(DPO_DIR) if os.path.exists(f'{DPO_DIR}/{d}/metrics.jsonl'))
rows = []
for run in runs:
    pair = {}
    for line in open(f'{DPO_DIR}/{run}/metrics.jsonl'):
        r = json.loads(line); pair[r['epoch']] = r
    for f in glob.glob(f'{DPO_DIR}/{run}/val_captions_epoch*.json'):
        e = int(re.search(r'epoch(-?\d+)\.json$', f).group(1))
        p, r = load(f)
        rows.append({'run': run, 'epoch': e, 'val_acc': 100 * pair[e]['val_acc'], 'val_loss': pair[e]['val_loss'],
                     **quick_scores(p, r), **extras(p)})
e5_val = pd.DataFrame(rows).sort_values(['run', 'epoch']).reset_index(drop=True)
e5_val['eligible'] = e5_val['distinct'] >= MIN_DISTINCT
e5_val.to_csv(f'{OUT_DIR}/e5_val_by_run_epoch.csv', index=False)
print(e5_val[['run', 'epoch', 'val_loss', 'val_acc'] + COLS + ['eligible']].round(3).to_string(index=False))

start = e5_val[e5_val['epoch'] == -1]['CIDEr'].mean()
cand = e5_val[(e5_val['epoch'] >= 0) & e5_val['eligible']]
best = cand.loc[cand['CIDEr'].idxmax()]
E5_BEST = (best['run'], int(best['epoch']))
print(f'\nstarting checkpoint (E4) val CIDEr: {start:.2f}')
print(f'best DPO run/epoch (fixed scores): {E5_BEST} val CIDEr {best["CIDEr"]:.2f}   | picked earlier: {E5_PICKED}')
if E5_BEST != E5_PICKED:
    print('-> A different run or epoch wins. Test captions exist only for the earlier pick, so E5 on test needs '
          f'kale_dpo_colab.ipynb rerun with SWEEP = [{E5_BEST[0]!r} settings] and epochs = {E5_BEST[1] + 1}.')

## 5. Test: E4 vs E5 with all metrics

Scores the test captions that exist (E4 epoch 5 and the earlier E5 pick). Qwen E1 is the team's
saved result, for reference. BERTScore downloads `roberta-large` the first time (about 1.4 GB).

In [ ]:
test = {}
for name, path in [(f'E4: KALE retrained (epoch {E4_PICKED_EPOCH})', f'{SFT_DIR}/test_captions_sft.json'),
                   (f'E5: KALE + DPO ({E5_PICKED[0]}, epoch {E5_PICKED[1]})', f'{DPO_DIR}/{E5_PICKED[0]}/test_captions.json')]:
    p, r = load(path)
    test[name] = {**full_scores(p, r), **extras(p)}
    print('scored', name)
qwen = json.load(open('data/baseline/qwen3_vl_baseline_metrics.json'))
test['E1: Qwen3-VL zero-shot (team result)'] = {k: 100 * v for k, v in qwen.items()}
TEST_COLS = ['Bleu_1', 'Bleu_2', 'Bleu_3', 'Bleu_4', 'METEOR', 'ROUGE_L', 'CIDEr', 'BERTScore_F1', 'mean_len', 'distinct']
test_table = pd.DataFrame(test).T.reindex(columns=TEST_COLS)
test_table.to_csv(f'{OUT_DIR}/test_E1_E4_E5.csv')
print(test_table.round(2).to_string())

## 6. The replication run, rescored

The replication trained on all 2,524 train+val paintings and captioned the test set after every epoch,
so it has no separate validation set: picking its best epoch here uses the test set and is only for
comparison with the paper.

In [ ]:
files = {int(m.group(1)): f for f in glob.glob('data/kale/result_epoch*.json')
         if (m := re.fullmatch(r'result_epoch(\d+)\.json', os.path.basename(f)))}
rows = {}
for e in sorted(files):
    p, r = load(files[e])
    rows[e] = {**quick_scores(p, r), **extras(p)}
rep = pd.DataFrame(rows).T[COLS]
rep.index.name = 'epoch'
rep.to_csv(f'{OUT_DIR}/replication_test_by_epoch.csv')
print(rep.iloc[[0, 1, 2, 5, 10, 20, 30, 40, 50, len(rep) - 1]].round(2).to_string())

last = len(rep) - 1
p, r = load(files[last])
last_full = full_scores(p, r)
vs = pd.DataFrame({f'ours, epoch {last} (fixed)': {k: last_full[k] for k in PAPER},
                   'KALE paper': PAPER})
vs['difference'] = vs.iloc[:, 0] - vs['KALE paper']
vs.to_csv(f'{OUT_DIR}/replication_vs_paper.csv')
print(f'\nLast epoch vs the KALE paper:')
print(vs.round(2).to_string())

## Save

The corrected tables are on Drive in `KALE_files/results_rescored/`. To add this to GitHub, download
this notebook with its outputs (File > Download > .ipynb), replace `baselines/kale_rescore_colab.ipynb`
with it, and copy the CSVs into `data/kale/rescored/`.

In [ ]:
!ls -la {OUT_DIR}